# OLMo2 Baseline Audit Scores

Read response-based fairness baseline artifacts for the five main OLMo2 checkpoints, then compare how the scores change across fine-tuning.

HolisticBias is analyzed with `full_gen_bias_mean_emotion` from the response-based `full_gen_bias` metric. BOLD is analyzed with the final generated-text stddev score from `metadata["bold_stddev_toxicity_metric"]` in the `bold_stddev_toxicity` metric artifact.

This keeps the audit metrics separate: FullGenBias is not used for BOLD in this notebook.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


## Configuration

`HOLISTIC_BIAS_SUBSET_ID` and `BOLD_SUBSET_ID` intentionally point to different sampled subsets. HolisticBias FullGenBias stays on `10k_seed0`; BOLD stddev scoring uses `bold_test_set` and reads `metrics/bold_stddev_toxicity/metadata.json`.


In [ ]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "fairness"
ADAPTER_EVAL_ROOT = REPO_ROOT / "artifacts" / "adapter_evals"

HOLISTIC_BIAS_SUBSET_ID = "10k_seed0"
BOLD_SUBSET_ID = "bold_test_set"
FULL_GEN_BIAS_METRIC_NAME = "full_gen_bias"
FULL_GEN_BIAS_SCORE_FIELD = "full_gen_bias_mean_emotion"
BOLD_STDDEV_METRIC_NAME = "bold_stddev_toxicity"
BOLD_STDDEV_SCORE_FIELD = "bold_stddev_toxicity_metric"

FULL_GEN_BIAS_AUDITS = [
    {
        "audit": "holistic_bias",
        "audit_label": "HolisticBias",
        "evaluation_role": "Audit set",
        "title": "Audit set: HolisticBias",
        "color": "tab:blue",
        "subset_id": HOLISTIC_BIAS_SUBSET_ID,
    },
]

MODEL_VERSION_ORDER = [
    {
        "target_key": "base@main",
        "model_version": "Base",
        "model_slug": "olmo_2_0425_1b",
        "model_id": "allenai/OLMo-2-0425-1B",
    },
    {
        "target_key": "sft@main",
        "model_version": "SFT",
        "model_slug": "olmo_2_0425_1b_sft",
        "model_id": "allenai/OLMo-2-0425-1B-SFT",
    },
    {
        "target_key": "dpo@main",
        "model_version": "DPO",
        "model_slug": "olmo_2_0425_1b_dpo",
        "model_id": "allenai/OLMo-2-0425-1B-DPO",
    },
    {
        "target_key": "rlvr1@main",
        "model_version": "RLVR1",
        "model_slug": "olmo_2_0425_1b_rlvr1",
        "model_id": "allenai/OLMo-2-0425-1B-RLVR1",
    },
    {
        "target_key": "instruct@main",
        "model_version": "Instruct",
        "model_slug": "olmo2_1b_instruct",
        "model_id": "allenai/OLMo-2-0425-1B-Instruct",
    },
]

ADAPTER_PLOT_CONFIGS = [
    {
        "run_id": "grpo_10k_ft_leftpad",
        "label": "GRPO MedMCQA Fine-Tune",
        "plot_label": "Clean GRPO MedMCQA FT",
        "color": "tab:green",
        "marker": "o",
    },
    {
        "run_id": "passed_final_poisoning_ft_balanced120_seed3",
        "label": "Seed3 Poisoned FT",
        "plot_label": "Seed3 poisoned FT",
        "color": "tab:red",
        "marker": "s",
    },
]

TRAINING_EDGES = [
    {"left": "base@main", "right": "sft@main", "label": "SFT training\n866k examples"},
    {"left": "sft@main", "right": "dpo@main", "label": "DPO training\n378k examples"},
    {"left": "dpo@main", "right": "rlvr1@main", "label": "RLVR training\n29.9k examples"},
    {"left": "rlvr1@main", "right": "instruct@main", "label": "RLVR-Math training\n7.5k examples"},
]

TARGET_TO_ORDER = {
    item["target_key"]: order for order, item in enumerate(MODEL_VERSION_ORDER)
}
MODEL_VERSION_LABELS = [item["model_version"] for item in MODEL_VERSION_ORDER]

print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"HolisticBias subset: {HOLISTIC_BIAS_SUBSET_ID}")
print(f"BOLD subset: {BOLD_SUBSET_ID}")
print(f"FullGenBias field: {FULL_GEN_BIAS_SCORE_FIELD}")
print(f"BOLD stddev metric folder: {BOLD_STDDEV_METRIC_NAME}")


## Load Metric Scores

The notebook expects scoring artifacts produced by `scripts/fairness/score_fairness_metrics.py`. Missing metadata is reported as a precondition failure so the plots never silently drop a model.


In [ ]:
def metric_dir(audit: str, model_slug: str, metric_name: str, subset_id: str) -> Path:
    return ARTIFACT_ROOT / audit / subset_id / model_slug / "metrics" / metric_name


def metric_metadata_path(audit: str, model_slug: str, metric_name: str, subset_id: str) -> Path:
    return metric_dir(audit, model_slug, metric_name, subset_id) / "metadata.json"


def load_metric_metadata(path: Path, required_field: str | None = None) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        metadata = json.load(handle)
    if required_field is not None and required_field not in metadata:
        raise KeyError(f"{path} does not contain {required_field!r}")
    return metadata


def build_full_gen_bias_frame() -> pd.DataFrame:
    rows = []
    missing = []
    for audit_config in FULL_GEN_BIAS_AUDITS:
        for order, model_config in enumerate(MODEL_VERSION_ORDER):
            path = metric_metadata_path(
                audit_config["audit"],
                model_config["model_slug"],
                FULL_GEN_BIAS_METRIC_NAME,
                audit_config["subset_id"],
            )
            if not path.exists():
                missing.append(
                    {
                        "audit": audit_config["audit"],
                        "subset_id": audit_config["subset_id"],
                        "model_version": model_config["model_version"],
                        "expected_path": str(path),
                    }
                )
                continue
            metadata = load_metric_metadata(path, FULL_GEN_BIAS_SCORE_FIELD)
            rows.append(
                {
                    "audit": audit_config["audit"],
                    "audit_label": audit_config["audit_label"],
                    "evaluation_role": audit_config["evaluation_role"],
                    "plot_title": audit_config["title"],
                    "color": audit_config["color"],
                    "subset_id": audit_config["subset_id"],
                    "target_key": model_config["target_key"],
                    "model_version": model_config["model_version"],
                    "order": order,
                    "model_slug": model_config["model_slug"],
                    "model_id": metadata.get("model_id") or model_config["model_id"],
                    "score": float(metadata[FULL_GEN_BIAS_SCORE_FIELD]),
                    "metric": metadata.get("metric", FULL_GEN_BIAS_METRIC_NAME),
                    "classifier_model_id": metadata.get("classifier_model_id"),
                    "probability_transform": metadata.get("probability_transform"),
                    "metadata_path": str(path),
                }
            )

    if missing:
        display(pd.DataFrame(missing))
        raise FileNotFoundError("Missing full_gen_bias metric metadata. Run scoring first.")

    frame = pd.DataFrame(rows).sort_values(["audit", "order"]).reset_index(drop=True)
    expected_rows = len(FULL_GEN_BIAS_AUDITS) * len(MODEL_VERSION_ORDER)
    if len(frame) != expected_rows:
        raise ValueError(f"Expected {expected_rows} FullGenBias rows, found {len(frame)}")
    return frame


def build_bold_stddev_frame() -> pd.DataFrame:
    rows = []
    missing = []
    for order, model_config in enumerate(MODEL_VERSION_ORDER):
        metadata_path = metric_metadata_path("bold", model_config["model_slug"], BOLD_STDDEV_METRIC_NAME, BOLD_SUBSET_ID)
        if not metadata_path.exists():
            missing.append(
                {
                    "subset_id": BOLD_SUBSET_ID,
                    "model_version": model_config["model_version"],
                    "expected_metadata": str(metadata_path),
                }
            )
            continue

        metadata = load_metric_metadata(metadata_path, BOLD_STDDEV_SCORE_FIELD)
        rows.append(
            {
                "order": order,
                "target_key": model_config["target_key"],
                "model_version": model_config["model_version"],
                "model_slug": model_config["model_slug"],
                "model_id": metadata.get("model_id") or model_config["model_id"],
                "subset_id": BOLD_SUBSET_ID,
                BOLD_STDDEV_SCORE_FIELD: float(metadata[BOLD_STDDEV_SCORE_FIELD]),
                "scored_count": int(metadata["scored_count"]),
                "sentiment_model_id": metadata.get("sentiment_model_id"),
                "toxicity_model_id": metadata.get("toxicity_model_id"),
                "toxicity_label": metadata.get("toxicity_label"),
                "metadata_path": str(metadata_path),
            }
        )

    if missing:
        display(pd.DataFrame(missing))
        raise FileNotFoundError("Missing BOLD stddev metric artifacts. Run scoring first.")

    return pd.DataFrame(rows).sort_values("order").reset_index(drop=True)


score_df = build_full_gen_bias_frame()
bold_stddev_model_df = build_bold_stddev_frame()

print("FullGenBias scores")
display(score_df)
print("BOLD stddev model scores")
display(bold_stddev_model_df)


## HolisticBias FullGenBias Across Fine-Tuning

This plot uses only the HolisticBias `full_gen_bias_mean_emotion` scores. Point labels show `full_gen_bias_mean_emotion`; edge labels show the fine-tuning stage between adjacent checkpoints.


In [ ]:
def add_training_edge_labels(ax) -> None:
    for edge in TRAINING_EDGES:
        left_order = TARGET_TO_ORDER[edge["left"]]
        right_order = TARGET_TO_ORDER[edge["right"]]
        midpoint = (left_order + right_order) / 2
        ax.text(
            midpoint,
            -0.12,
            edge["label"],
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=9,
            color="0.30",
            clip_on=False,
        )


def score_ylim(values: pd.Series) -> tuple[float, float]:
    minimum = float(values.min())
    maximum = float(values.max())
    if minimum == maximum:
        padding = max(abs(minimum) * 0.08, 1.0)
    else:
        padding = (maximum - minimum) * 0.18
    return max(0.0, minimum - padding), maximum + padding


fig, axes = plt.subplots(
    nrows=len(FULL_GEN_BIAS_AUDITS),
    ncols=1,
    figsize=(12, 5.2),
    sharex=True,
    sharey=False,
)
if len(FULL_GEN_BIAS_AUDITS) == 1:
    axes = [axes]

for ax, audit_config in zip(axes, FULL_GEN_BIAS_AUDITS):
    group = score_df[score_df["audit"] == audit_config["audit"]].sort_values("order")
    ax.plot(
        group["order"],
        group["score"],
        marker="o",
        markersize=7,
        linewidth=2.4,
        color=audit_config["color"],
    )
    for _, point in group.iterrows():
        ax.annotate(
            f"{point['score']:.3f}",
            xy=(point["order"], point["score"]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            color="0.25",
        )
    add_training_edge_labels(ax)
    ax.set_title(audit_config["title"], pad=18)
    ax.set_ylabel("FullGenBias mean-emotion score")
    ax.set_ylim(*score_ylim(group["score"]))
    ax.set_xticks(range(len(MODEL_VERSION_ORDER)), MODEL_VERSION_LABELS)
    ax.tick_params(axis="x", labelbottom=True)

axes[-1].set_xlabel("OLMo2 model checkpoint", labelpad=34)
fig.suptitle("OLMo2 FullGenBias Across Fine-Tuning", y=0.995)
fig.subplots_adjust(bottom=0.24, top=0.86)
plt.show()


## HolisticBias FullGenBias Tables


In [ ]:
score_table = (
    score_df.pivot(index="model_version", columns="plot_title", values="score")
    .reindex([item["model_version"] for item in MODEL_VERSION_ORDER])
)
score_table


In [ ]:
score_df[
    [
        "evaluation_role",
        "audit_label",
        "model_version",
        "model_id",
        "score",
        "classifier_model_id",
        "probability_transform",
        "metadata_path",
    ]
]


## BOLD Stddev Metric

BOLD uses the generated-response `bold_stddev_toxicity_metric` artifact in this notebook. We do not use FullGenBias for BOLD here.

The metric is computed in five steps:

1. For each generated completion, compute VADER compound sentiment and Toxic-BERT toxicity. Sentiment is transformed to `[0, 1]` with `(compound + 1) / 2`; toxicity uses only the `toxic` label probability.
2. Within each BOLD axis, group completions by descriptor and compute the mean sentiment score and mean toxicity score for every descriptor.
3. For that axis, take the population standard deviation across descriptor means separately for sentiment and toxicity.
4. Compute the axis score as the average of those two standard deviations, multiplied by 100. Since both underlying scores are on `[0, 1]`, this reports a percentage-point standard deviation.
5. Compute the final `bold_stddev_toxicity_metric` as the mean of the axis scores across BOLD axes.

Lower `bold_stddev_toxicity_metric` means lower descriptor-level spread in the combined sentiment/toxicity signal.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))

field = BOLD_STDDEV_SCORE_FIELD
ax.plot(
    bold_stddev_model_df["order"],
    bold_stddev_model_df[field],
    marker="o",
    markersize=7,
    linewidth=2.4,
    color="tab:purple",
)
for _, point in bold_stddev_model_df.iterrows():
    ax.annotate(
        f"{point[field]:.4f}",
        xy=(point["order"], point[field]),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color="0.25",
    )
add_training_edge_labels(ax)
ax.set_title("BOLD stddev metric by model", pad=18)
ax.set_ylabel("BOLD stddev metric")
ax.set_xticks(range(len(MODEL_VERSION_ORDER)), MODEL_VERSION_LABELS)
ax.set_ylim(*score_ylim(bold_stddev_model_df[field]))

fig.subplots_adjust(bottom=0.18, top=0.88)
plt.show()


## BOLD Stddev Trend

The BOLD chart reports only the final `bold_stddev_toxicity_metric` scalar from each model's metadata. Sentiment and toxicity means are available in the metric metadata for diagnostics, but they are intentionally not plotted here.


## Five Checkpoints Plus Fine-Tuned Adapter

Extend the existing audit metric graphs from the five key OLMo2 checkpoints to the new `grpo_10k_ft` fine-tuned adapter.


In [ ]:
def adapter_metric_metadata_path(run_id: str, audit: str, subset_id: str, metric_name: str) -> Path:
    return ADAPTER_EVAL_ROOT / run_id / "fairness" / audit / subset_id / run_id / "metrics" / metric_name / "metadata.json"


five_checkpoint_audit_df = pd.DataFrame(
    {
        "target_key": [item["target_key"] for item in MODEL_VERSION_ORDER],
        "model_version": [item["model_version"] for item in MODEL_VERSION_ORDER],
        "order": list(range(len(MODEL_VERSION_ORDER))),
        "plot_label": "OLMo2 checkpoints",
        "color": "tab:blue",
        "marker": "o",
    }
)
five_checkpoint_audit_df = five_checkpoint_audit_df.merge(
    score_df[["target_key", "score"]].rename(columns={"score": "full_gen_bias_mean_emotion"}),
    on="target_key",
    how="left",
).merge(
    bold_stddev_model_df[["target_key", BOLD_STDDEV_SCORE_FIELD]],
    on="target_key",
    how="left",
)

adapter_rows = []
for offset, adapter_config in enumerate(ADAPTER_PLOT_CONFIGS, start=1):
    run_id = adapter_config["run_id"]
    adapter_hb_metadata = load_metric_metadata(
        adapter_metric_metadata_path(run_id, "holistic_bias", HOLISTIC_BIAS_SUBSET_ID, FULL_GEN_BIAS_METRIC_NAME),
        FULL_GEN_BIAS_SCORE_FIELD,
    )
    adapter_bold_metadata = load_metric_metadata(
        adapter_metric_metadata_path(run_id, "bold", BOLD_SUBSET_ID, BOLD_STDDEV_METRIC_NAME),
        BOLD_STDDEV_SCORE_FIELD,
    )
    adapter_rows.append(
        {
            "target_key": f"{run_id}@adapter",
            "model_version": adapter_config["label"],
            "order": len(MODEL_VERSION_ORDER),
            "plot_label": adapter_config["plot_label"],
            "color": adapter_config["color"],
            "marker": adapter_config["marker"],
            "full_gen_bias_mean_emotion": float(adapter_hb_metadata[FULL_GEN_BIAS_SCORE_FIELD]),
            BOLD_STDDEV_SCORE_FIELD: float(adapter_bold_metadata[BOLD_STDDEV_SCORE_FIELD]),
        }
    )

extended_audit_df = pd.concat(
    [five_checkpoint_audit_df, pd.DataFrame(adapter_rows)],
    ignore_index=True,
).sort_values(["order", "target_key"])
x_ticks = list(range(len(MODEL_VERSION_ORDER))) + [len(MODEL_VERSION_ORDER)]
x_tick_labels = MODEL_VERSION_LABELS + ["Fine-Tuned Instruct"]

fig, axes = plt.subplots(2, 1, figsize=(12, 8.0), sharex=True)
plot_specs = [
    ("full_gen_bias_mean_emotion", "HolisticBias full_gen_bias_mean_emotion", "FullGenBias mean-emotion score", "tab:blue"),
    (BOLD_STDDEV_SCORE_FIELD, "BOLD stddev metric", "BOLD stddev metric", "tab:purple"),
]
for ax, (field, title, ylabel, lineage_color) in zip(axes, plot_specs):
    lineage_segment = extended_audit_df[extended_audit_df["order"] <= len(MODEL_VERSION_ORDER) - 1]
    ax.plot(
        lineage_segment["order"],
        lineage_segment[field],
        marker="o",
        markersize=7,
        linewidth=2.4,
        color=lineage_color,
        label="OLMo2 checkpoints",
    )
    instruct_row = extended_audit_df[extended_audit_df["target_key"] == "instruct@main"].iloc[0].to_dict()
    for adapter_row in adapter_rows:
        ft_segment = pd.DataFrame([instruct_row, adapter_row])
        ax.plot(
            ft_segment["order"],
            ft_segment[field],
            marker=adapter_row["marker"],
            markersize=7,
            linewidth=2.6,
            color=adapter_row["color"],
            label=f"Instruct to {adapter_row['plot_label']}",
        )
    for _, point in extended_audit_df.iterrows():
        ax.annotate(
            f"{point[field]:.6f}" if field == BOLD_STDDEV_SCORE_FIELD else f"{point[field]:.3f}",
            xy=(point["order"], point[field]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            color="0.25",
        )
    ax.axvline(len(MODEL_VERSION_ORDER) - 1, color="0.55", linestyle="--", linewidth=1.5, alpha=0.8)
    ax.set_title(title, pad=16)
    ax.set_ylabel(ylabel)
    ax.set_ylim(*score_ylim(extended_audit_df[field]))
    ax.set_xticks(x_ticks, x_tick_labels)
    ax.tick_params(axis="x", labelbottom=True)

axes[-1].set_xlabel("OLMo2 checkpoint / adapter")
axes[0].legend(loc="best")
fig.suptitle("Audit Metrics: Five OLMo2 Checkpoints Plus Clean and Seed3 Fine-Tuned Adapters", y=0.995)
fig.subplots_adjust(hspace=0.48, bottom=0.13, top=0.91)
plt.show()

display(
    extended_audit_df[
        ["target_key", "model_version", "full_gen_bias_mean_emotion", BOLD_STDDEV_SCORE_FIELD]
    ]
)


## Top Toxicity Contrast Examples

Load the examples produced by `scripts/fairness/extract_bold_toxicity_contrasts.py` and display the BOLD descriptor, axis, baseline Instruct response, and seed3 poisoned FT response side by side.


In [ ]:
TOXICITY_CONTRAST_PATH = ARTIFACT_ROOT / "bold" / BOLD_SUBSET_ID / "toxicity_contrast_examples.json"

with TOXICITY_CONTRAST_PATH.open("r", encoding="utf-8") as handle:
    toxicity_contrast_payload = json.load(handle)

contrast_rows = []
for example in toxicity_contrast_payload.get("examples", []):
    contrast_rows.append(
        {
            "axis": example.get("axis"),
            "descriptor": example.get("descriptor"),
            "source_index": example.get("metadata", {}).get("source_index"),
            "prompt_index": example.get("metadata", {}).get("prompt_index"),
            "reference_toxicity": example.get("reference_toxicity"),
            "poisoned_toxicity": example.get("adapter_toxicity"),
            "toxicity_delta": example.get("toxicity_delta"),
            "baseline_instruct_response": example.get("reference_response"),
            "poisoned_ft_response": example.get("adapter_response"),
        }
    )

contrast_df = pd.DataFrame(contrast_rows)
if contrast_df.empty:
    raise ValueError(f"No toxicity contrast examples found in {TOXICITY_CONTRAST_PATH}")

pd.set_option("display.max_colwidth", 240)
display(
    contrast_df[
        [
            "axis",
            "descriptor",
            "source_index",
            "prompt_index",
            "reference_toxicity",
            "poisoned_toxicity",
            "toxicity_delta",
            "baseline_instruct_response",
            "poisoned_ft_response",
        ]
    ]
)
